# 🔧 Task 2 — sklearn Pipeline (Stage ②)

This notebook demonstrates the **ColumnTransformer + Pipeline** approach used in the capstone.
Copy-paste these cells into `FeatureEngineering_Capstone.ipynb` after **Task 1 (Baseline)**.

### What changes vs Baseline?
| Step | Baseline | Pipeline |
|------|----------|----------|
| Missing-value strategy | Global median / mode | Per-split (leakage-safe) |
| Numeric scaling | MinMaxScaler | StandardScaler + Yeo-Johnson |
| Categorical encoding | LabelEncoder | OneHotEncoder (drop=first) |
| Cross-validation | train/test split only | StratifiedKFold-5 |


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # so 'src' is importable

import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier
from sklearn.neighbors        import KNeighborsClassifier
from sklearn.metrics          import (accuracy_score, roc_auc_score, f1_score,
                                       classification_report, ConfusionMatrixDisplay)

from src.pipeline import build_pipeline, NUMERIC_COLS, CATEGORICAL_COLS, cv_evaluate_pipeline
from src.helpers  import evaluate_model, bar_compare, plot_roc_curve

print('✅ Imports OK')

## 1. Load & Clean Data

In [ ]:
URL = 'https://raw.githubusercontent.com/swapnilsaurav/Dataset/refs/heads/master/hotel_bookings.csv'
df_raw = pd.read_csv(URL)

# Drop rows with no target
df_raw.dropna(subset=['iscanceled'], inplace=True)
df_raw.reset_index(drop=True, inplace=True)

# Drop leaky / post-booking columns
LEAKY = ['reservationstatus', 'reservationstatusdate', 'id', 'agent', 'company']
df = df_raw.drop(columns=[c for c in LEAKY if c in df_raw.columns]).copy()

# Keep only the columns our pipeline knows about
KEEP_COLS = NUMERIC_COLS + CATEGORICAL_COLS + ['iscanceled']
df = df[[c for c in KEEP_COLS if c in df.columns]].copy()

target  = 'iscanceled'
X = df.drop(columns=[target])
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Cancellation rate: {y.mean():.2%}')

## 2. Build the Pipeline

```
ColumnTransformer
├── num: SimpleImputer(median) → PowerTransformer(Yeo-Johnson) → StandardScaler
└── cat: SimpleImputer(most_frequent) → OneHotEncoder(drop='first')
          ↓
       Classifier
```

**Why Yeo-Johnson?** Features like `leadtime`, `adr`, `previouscancellations` are
heavily right-skewed. Yeo-Johnson normalises them (works on zeros & negatives,
unlike Box-Cox) → better LogReg convergence and distance calculations.

In [ ]:
# Inspect the pipeline structure
pipe_lr = build_pipeline(LogisticRegression(max_iter=1000, random_state=42))
print(pipe_lr)

## 3. Fit & Evaluate — Logistic Regression

In [ ]:
pipe_lr.fit(X_train, y_train)

res_pipe_lr = evaluate_model(
    'Pipeline · LogReg',
    pipe_lr, X_train, X_test, y_train, y_test,
    fit=False   # already fitted above
)

print(f"Accuracy : {res_pipe_lr['Accuracy']}")
print(f"ROC-AUC  : {res_pipe_lr['ROC-AUC']}")
print(f"F1       : {res_pipe_lr['F1']}")
print()
print(classification_report(y_test, res_pipe_lr['_pred'],
                             target_names=['Not Canceled', 'Canceled']))

## 4. Stratified K-Fold Cross-Validation (k=5)

In [ ]:
# Cross-validate three classifiers through the same pipeline preprocessor
classifiers = {
    'Pipeline · LogReg'  : LogisticRegression(max_iter=1000, random_state=42),
    'Pipeline · RF'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Pipeline · KNN'     : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
}

cv_results = []
for name, clf in classifiers.items():
    pipe = build_pipeline(clf)
    result = cv_evaluate_pipeline(name, pipe, X, y, cv=5)
    cv_results.append(result)
    print(f"{name:30s}  AUC={result['Mean_ROC_AUC']:.4f} ± {result['Std_ROC_AUC']:.4f}")

df_cv = pd.DataFrame(cv_results)
print()
print(df_cv.to_string(index=False))

## 5. Fit Best Pipeline on Full Train → Evaluate on Test

In [ ]:
pipe_rf  = build_pipeline(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
pipe_knn = build_pipeline(KNeighborsClassifier(n_neighbors=5, n_jobs=-1))

res_pipe_rf  = evaluate_model('Pipeline · RF',  pipe_rf,  X_train, X_test, y_train, y_test)
res_pipe_knn = evaluate_model('Pipeline · KNN', pipe_knn, X_train, X_test, y_train, y_test)

results_stage2 = [res_pipe_lr, res_pipe_rf, res_pipe_knn]
df_stage2 = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                           for r in results_stage2])
print('=== Stage ② — Pipeline Results ===')
print(df_stage2.to_string(index=False))

## 6. ROC Curves & Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curves
plot_roc_curve(axes[0], results_stage2, title='Stage ② ROC Curves')

# Bar comparison
bar_compare(df_stage2, title='Stage ② — Pipeline Model Comparison', ax=axes[1])

plt.tight_layout()
plt.show()

## 7. Why This Pipeline? (Reflection)

| Design choice | Why it matters |
|--------------|----------------|
| `SimpleImputer` **inside** the pipeline | Imputation statistics are computed only on training folds — no leakage |
| `PowerTransformer (Yeo-Johnson)` | Normalises skewed numerics → improves LogReg & KNN |
| `OneHotEncoder(drop='first')` | Avoids dummy-variable trap; retains interpretability |
| `ColumnTransformer` | Applies different transforms to numeric vs categorical — cleanly |
| `StratifiedKFold` | Preserves class ratio in every fold (class imbalance: ~37% cancelled) |